# Extract a small sample table incl. different years

In [1]:
import pandas as pd

In [7]:
# ---------- Setup ----------

# Define years
years = list(range(2002, 2019))  # 2002-2018 inclusive

# Paths to input and output files
parquet_paths = {y: f"/Users/Wanja/Documents/non-equilibrium_data/cv_new/table_wgs84_{y}_with_cv.parquet" for y in years}
output_path = "/Users/Wanja/Documents/non-equilibrium_data/cv_new/table_wgs84_2002-2018_with_cv_sample.parquet"

In [3]:
# ---------- Concatenate all yearly tables ----------

# round coordinates to avoid floating point issues when grouping (just to be safe, they already contain 6 decimal places)
COORD_DECIMALS = 6

dfs = []
for y, p in parquet_paths.items():
    d = pd.read_parquet(p)  # load all variables/columns
    d["lon_r"] = d["longitude"].round(COORD_DECIMALS)
    d["lat_r"] = d["latitude"].round(COORD_DECIMALS)
    # "year" column already exists in the data, so we don't overwrite it
    dfs.append(d)

long_df = pd.concat(dfs, ignore_index=True)

# sanity check: every pixel should have all 17 years
counts = long_df.groupby(["lon_r", "lat_r"])["year"].nunique()
n_incomplete = (counts != len(years)).sum()
print(f"{n_incomplete} pixels missing at least one year (out of {len(counts)})")

# Check for duplicate (lon, lat, year) combinations
dupes = long_df.duplicated(subset=["lon_r", "lat_r", "year"]).sum()
print(f"{dupes} duplicate lon/lat/year rows")

# Check total row count makes sense: unique_pixels * 17 years
n_pixels = long_df.groupby(["lon_r", "lat_r"]).ngroups
print(f"{n_pixels} unique pixels, expected total rows: {n_pixels * len(years)}, actual: {len(long_df)}")

0 pixels missing at least one year (out of 4218230)
0 duplicate lon/lat/year rows
4218230 unique pixels, expected total rows: 71709910, actual: 71709910


In [4]:
# ---------- Randomly sample 100,000 pixels, keep all years for those pixels ----------

N_PIXELS_SAMPLE = 100_000
RANDOM_SEED = 42  # set to None if you want a different sample every run

# unique pixel (lon_r, lat_r) combinations
unique_pixels = long_df[["lon_r", "lat_r"]].drop_duplicates().reset_index(drop=True)

n_available = len(unique_pixels)
if N_PIXELS_SAMPLE > n_available:
    raise ValueError(
        f"Requested {N_PIXELS_SAMPLE} pixels but only {n_available} unique pixels are available."
    )

sampled_pixels = unique_pixels.sample(n=N_PIXELS_SAMPLE, random_state=RANDOM_SEED)

# keep all years for the sampled pixels via inner merge
sample_df = long_df.merge(sampled_pixels, on=["lon_r", "lat_r"], how="inner")

print(f"Sampled {sampled_pixels.shape[0]} pixels, resulting in {sample_df.shape[0]} rows "
      f"(expected up to {sampled_pixels.shape[0] * len(years)} if all pixels have all years).")

Sampled 100000 pixels, resulting in 1700000 rows (expected up to 1700000 if all pixels have all years).


In [8]:
sample_df_to_save = sample_df.drop(columns=["lon_r", "lat_r"])
sample_df_to_save.to_parquet(output_path, index=False)
print(f"Saved {sample_df_to_save.shape[0]} rows to {output_path}")

Saved 1700000 rows to /Users/Wanja/Documents/non-equilibrium_data/cv_new/table_wgs84_2002-2018_with_cv_sample.parquet
